In [10]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

In [11]:
# Making Custom Maths for the AdaBoosting

In [24]:
class Custom_AdaBoost:

    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.models = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Initialize sample weights
        w = np.ones(n_samples) / n_samples
        for i in range(self.n_estimators):
            model = DecisionTreeClassifier(max_depth=1, random_state=42)
            model.fit(X, y, sample_weight=w)
            # Predictions on training data
            predictions = model.predict(X)
            # Weighted Error
            error = np.sum(w * (predictions != y)) / np.sum(w)
            # Avoid division by zero
            error = np.clip(error, 1e-10, 1 - 1e-10)
            # Compute Alpha (Model Weight)
            alpha = 0.5 * np.log((1 - error) / error)
            # Store model and alpha
            self.models.append(model)
            self.alphas.append(alpha)
            # Update sample weights
            w *= np.exp(-alpha * y * predictions)
            # Normalize
            w /= np.sum(w)

    def predict(self, X):
        strong_predictions = np.zeros(X.shape[0])
        # Weighted Voting
        for model, alpha in zip(self.models, self.alphas):
            predictions = model.predict(X)
            strong_predictions += alpha * predictions
        return np.sign(strong_predictions).astype(int)

In [25]:
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    random_state=42
)

# Convert labels from {0,1} to {-1,+1}
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
adaboost = Custom_AdaBoost(n_estimators=50)
adaboost.fit(X_train, y_train)
predictions = adaboost.predict(X_test)

print("Accuracy :", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall   :", recall_score(y_test, predictions))
print("F1 Score :", f1_score(y_test, predictions))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

Accuracy : 0.825
Precision: 0.8
Recall   : 0.8571428571428571
F1 Score : 0.8275862068965517

Confusion Matrix
[[81 21]
 [14 84]]
